# Attention Residuals 原理复现（中文版）

这个 notebook 面向初学者，尽量用中文解释每一步在做什么。

它构建了一个 6 层的小型因果 Transformer，并对比两种残差机制：

- 标准 PreNorm 残差累加
- 基于深度注意力的残差聚合（AttnRes）

目标是在个人电脑上复现论文的核心机制，而不是复现论文的大规模训练结果。


## 先理解这份 notebook 在做什么

可以把它理解成一个小实验：

1. 先造一批简单的序列数据
2. 用同样大小的两个模型分别训练
3. 比较它们的 loss、隐藏状态大小、以及 AttnRes 如何选择历史层

你不需要先完全懂 Transformer，也能先从图和注释里理解这个机制。


## 第 1 步：导入依赖

- `torch`：PyTorch，负责张量计算和训练
- `src.attnres_toy`：我们自己写的小实验代码
- `Path` 和 `sys`：让 notebook 能找到项目里的 `src` 目录


In [ ]:
from pathlib import Path
import sys
import time

# 让 notebook 能正确导入上一级目录中的 src/ 代码
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
from src.attnres_toy import (
    Config,
    TinyTransformerLM,
    collect_depth_statistics,
    evaluate_loss,
    plot_attnres_heatmap,
    plot_hidden_norms,
    plot_loss_curves,
    run_training,
    set_seed,
)


In [ ]:
figures_dir = project_root / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

loss_curve_path = figures_dir / "loss_curve.png"
hidden_norms_path = figures_dir / "hidden_norms.png"
attnres_heatmap_path = figures_dir / "attnres_heatmap.png"


## 第 2 步：设置实验参数

下面这些参数非常重要，含义如下：

- `device`：训练设备；有 GPU 就用 GPU，没有就用 CPU
- `vocab_size`：词表大小；这里不是自然语言词表，而是合成序列里的 token 种类数
- `seq_len`：每条序列的长度
- `d_model`：每个 token 的隐藏向量维度；越大模型表达能力越强，但更慢
- `n_heads`：多头注意力里的头数
- `d_ff`：前馈网络隐藏层大小；通常比 `d_model` 更大
- `n_layers`：Transformer 层数；这里设成 6 层，用来模拟论文里的“沿深度聚合”
- `steps`：训练多少步
- `batch_size`：每一步同时训练多少条样本
- `set_seed(42)`：固定随机种子，方便你多次运行得到相近结果

如果你电脑较慢，可以先把 `steps` 改小，比如改成 `40`。


In [ ]:
# 自动选择设备：有 CUDA 就用 GPU，否则用 CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# 配置一个很小的 Transformer，保证个人电脑也能跑
config = Config(
    vocab_size=32,   # 一共有 32 种 token
    seq_len=24,      # 每条输入序列长度是 24
    d_model=96,      # 每个 token 对应 96 维隐藏向量
    n_heads=4,       # 注意力头数是 4
    d_ff=192,        # 前馈网络中间层大小
    n_layers=6,      # Transformer 总层数
)

# 训练参数
steps = 120         # 训练 120 个 step
batch_size = 32     # 每个 step 用 32 条样本

# 固定随机种子，便于复现实验
set_seed(42)

# 打印当前实验配置，方便确认参数
print({"device": device, "config": config, "steps": steps, "batch_size": batch_size})


## 第 3 步：训练 baseline 和 AttnRes

这里使用一个带噪声的短序列 next-token prediction 合成任务，用来观察残差机制差异。

这一段做了三件事：

1. 初始化一个普通残差模型 `baseline`
2. 初始化一个 AttnRes 模型 `attnres`
3. 分别训练，然后把训练历史保存下来

注意：我在两个模型初始化前都重新设置了随机种子，这样两边初始条件更接近，便于公平对比。


In [ ]:
# 先初始化 baseline 模型（标准残差）
set_seed(42)
baseline = TinyTransformerLM(config, variant="baseline")

# 再初始化 AttnRes 模型（深度注意力残差）
set_seed(42)
attnres = TinyTransformerLM(config, variant="attnres")

# 分别训练两个模型
baseline_history = run_training(baseline, steps=steps, batch_size=batch_size, device=device)
attnres_history = run_training(attnres, steps=steps, batch_size=batch_size, device=device)

# 把历史结果放在一起，后面画图更方便
histories = {"baseline": baseline_history, "attnres": attnres_history}


## 第 4 步：看训练 loss 曲线

- 横轴：训练 step
- 纵轴：交叉熵损失（cross entropy）
- 一般来说越低越好

这里的目标不是一定让 AttnRes 的 loss 明显低很多，而是观察它是否能在小实验里表现出可学习性。


In [ ]:
# 画出两个模型的训练 loss 曲线，并保存到 figures/
plot_loss_curves(histories, save_path=loss_curve_path)


## 第 5 步：对比不同深度的隐藏状态范数

论文的一个关键观点是：标准残差会把很多层的输出不断累加，可能导致隐藏状态幅值随着深度不断变大。

这里的 `hidden_norms` 可以粗略理解为：

- 每一层输入表示的“平均大小”
- 数值越大，说明表示幅值越大
- 如果曲线随层数增长很快，就说明深度累积更明显


In [ ]:
# 收集两个模型在不同层上的统计信息
baseline_stats = collect_depth_statistics(baseline, device=device)
attnres_stats = collect_depth_statistics(attnres, device=device)

# 合并后用于统一画图
stats = {"baseline": baseline_stats, "attnres": attnres_stats}

# 画出不同层的隐藏状态范数，并保存到 figures/
plot_hidden_norms(stats, save_path=hidden_norms_path)

# 直接打印数值，方便你精确比较
baseline_stats["hidden_norms"], attnres_stats["hidden_norms"]


## 第 6 步：可视化 AttnRes 的深度选择权重

这张热力图是整份实验里最关键的一张图。

- 纵轴：当前是第几层 Transformer
- 横轴：它在看哪个历史深度状态
- 颜色越亮：说明该层越偏向使用那个历史状态

如果是普通残差，本质上更接近“统一累加”；而 AttnRes 则会学着“有选择地看以前哪些层更重要”。


In [ ]:
# 画出 AttnRes 在深度方向上的选择热力图，并保存到 figures/
plot_attnres_heatmap(attnres_stats, save_path=attnres_heatmap_path)

# 同时打印具体权重矩阵
attnres_stats["attn_depth_weights"]


## 重点观察什么

- AttnRes 会对不同历史深度状态学出非均匀权重，而不是平均相加。
- 相比 baseline，AttnRes 的隐藏状态范数通常更平缓。
- 即使 loss 接近，也能从热力图和范数曲线里看到机制差异。

如果你是初学者，可以先只看两张图：

1. 隐藏状态范数曲线
2. AttnRes 热力图

这两张图最能帮助你理解论文想解决的问题。


## 第 7 步：用多随机种子验证“是否真的提升”

前面的图能帮你理解机制，但还不能回答一个关键问题：

**AttnRes 在这个 toy 任务上，是否稳定优于 baseline？**

这里我们做一个 MVP 统计验证（兼顾本地电脑性能）：

- 用 3 个不同随机种子重复训练
- 记录训练耗时（秒）
- 在同样的评估批次上比较平均 eval loss

判断原则（简化版）：

- `eval loss` 更低，通常表示效果更好
- 如果提升 < 1%，通常不能算“显著提升”
- 如果速度明显变慢，要考虑性能-效果是否划算


In [ ]:
# 为了本地可运行，这里用小规模多次重复（MVP）
mvp_steps = 100
mvp_batch_size = 32
mvp_eval_batches = 10
mvp_seeds = [11, 22, 33]

def run_mvp_compare(variants):
    summary = {}
    for variant in variants:
        eval_losses = []
        train_times = []
        for seed in mvp_seeds:
            set_seed(seed)
            model = TinyTransformerLM(config, variant=variant)
            t0 = time.perf_counter()
            run_training(model, steps=mvp_steps, batch_size=mvp_batch_size, device=device)
            train_times.append(time.perf_counter() - t0)
            eval_losses.append(evaluate_loss(model, batches=mvp_eval_batches, batch_size=mvp_batch_size, device=device))

        mean_eval = sum(eval_losses) / len(eval_losses)
        mean_time = sum(train_times) / len(train_times)
        summary[variant] = {
            "mean_eval_loss": mean_eval,
            "mean_train_sec": mean_time,
            "raw_eval_losses": eval_losses,
        }

    baseline_loss = summary["baseline"]["mean_eval_loss"]
    baseline_time = summary["baseline"]["mean_train_sec"]
    for variant in variants:
        summary[variant]["improve_vs_baseline_percent"] = (baseline_loss - summary[variant]["mean_eval_loss"]) / baseline_loss * 100
        summary[variant]["time_overhead_vs_baseline_percent"] = (summary[variant]["mean_train_sec"] - baseline_time) / baseline_time * 100
    return summary

mvp_summary = run_mvp_compare(["baseline", "attnres", "depthcross_lite"])

print("variant | mean_eval_loss | improve_vs_baseline | mean_train_sec | time_overhead")
for name in ["baseline", "attnres", "depthcross_lite"]:
    item = mvp_summary[name]
    print(f"{name:14s} | {item['mean_eval_loss']:.4f} | {item['improve_vs_baseline_percent']:+.2f}% | {item['mean_train_sec']:.2f}s | {item['time_overhead_vs_baseline_percent']:+.1f}%")

mvp_summary


## 第 8 步：怎么理解这个结果

你可以按下面顺序看：

1. **先看 baseline vs attnres 的平均 eval loss**：是否真的下降
2. **再看提升百分比**：如果只有 0.x%，通常不算显著
3. **最后看训练时间开销**：如果更慢很多，且收益很小，工程上通常不划算

在这个 toy 复现里，常见现象是：

- AttnRes 机制是可见的（热力图确实显示了非均匀深度选择）
- 但“效果显著提升”不稳定，往往依赖任务、训练规模和超参数

这和论文结论并不矛盾：论文是在更大模型、更长训练和更真实任务上统计得到提升。
我们的 notebook 更适合验证“机制是否成立”，不适合直接当成“性能结论复现”。
